## V1.1: Build and Save the Full-Year Dataset

Load the 12 monthly BTS CSVs using only the columns needed for our target and scheduled-flight features. Remove rows with missing targets, optimize data types, engineer the V1 features, and save the processed dataset as Parquet so future sessions can load it quickly.

All model features must be available at the T-2-hour prediction cutoff. We will preserve the original raw files and record row counts, file size, and processing time.

In [1]:
import pandas as pd
import numpy as np
import time
from pathlib import Path

start_time = time.perf_counter()

# 1. Define paths
raw_dir = Path("../data/raw/2024")
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

output_path = processed_dir / "flights_2024.parquet"

# 2. Select only the columns needed
required_columns = [
    "FlightDate",
    "DepDel15",
    "CRSDepTime",
    "Month",
    "Reporting_Airline",
    "Origin",
    "Dest",
    "Distance",
    "CRSElapsedTime",
]

csv_files = sorted(raw_dir.glob("month_*/*.csv"))
assert len(csv_files) == 12, (
    f"Expected 12 monthly CSVs, found {len(csv_files)}"
)

# 3. Load and combine the monthly CSVs
frames = []

for file in csv_files:
    month_start = time.perf_counter()

    month_df = pd.read_csv(
        file,
        usecols=required_columns,
        low_memory=False
    )

    print(
        f"{file.parent.name}: {len(month_df):,} rows "
        f"in {time.perf_counter() - month_start:.1f}s"
    )

    frames.append(month_df)

df = pd.concat(frames, ignore_index=True)
del frames

rows_before = len(df)

print(f"\nCombined rows: {rows_before:,}")
print(f"Selected columns: {df.shape[1]}")
print(
    f"Initial DataFrame memory: "
    f"{df.memory_usage(deep=True).sum() / 1024**3:.2f} GB"
)

# 4. Clean the target and dates
df["FlightDate"] = pd.to_datetime(df["FlightDate"], errors="coerce")
df["Month"] = pd.to_numeric(df["Month"], errors="coerce")

missing_target_by_month = (
    df.loc[df["DepDel15"].isna()]
      .groupby("Month")
      .size()
      .reindex(range(1, 13), fill_value=0)
)

print("\nRows with missing target by month:")
print(missing_target_by_month)

df = df.dropna(subset=["DepDel15", "FlightDate"]).copy()
df["DepDel15"] = df["DepDel15"].astype("int8")

print(f"\nRows removed: {rows_before - len(df):,}")
print(f"Rows after cleaning: {len(df):,}")

# 5. Optimize data types
for col in ["Reporting_Airline", "Origin", "Dest"]:
    df[col] = df[col].astype("category")

for col in ["Month", "CRSDepTime"]:
    df[col] = pd.to_numeric(df[col], errors="coerce", downcast="integer")

for col in ["Distance", "CRSElapsedTime"]:
    df[col] = pd.to_numeric(df[col], errors="coerce", downcast="float")

# 6. Rebuild V1 scheduled-flight features
df["DepHour"] = df["CRSDepTime"] // 100
df["DepMinute"] = df["CRSDepTime"] % 100

df["FlightDay"] = df["FlightDate"].dt.day.astype("int8")
df["FlightMonth"] = df["FlightDate"].dt.month.astype("int8")
df["FlightDayOfWeek"] = df["FlightDate"].dt.dayofweek.astype("int8")

df["Route"] = (
    df["Origin"].astype("string")
    + "_"
    + df["Dest"].astype("string")
).astype("category")

df["ScheduledDuration"] = df["CRSElapsedTime"]

df["DistancePerMinute"] = (
    df["Distance"]
    / df["ScheduledDuration"].replace(0, np.nan)
)

df["DepTimeBlock"] = pd.cut(
    df["DepHour"],
    bins=[-1, 5, 11, 17, 23],
    labels=["Night", "Morning", "Afternoon", "Evening"]
).astype("category")

# 7. Define and check model features
features = [
    "FlightMonth",
    "FlightDay",
    "FlightDayOfWeek",
    "DepHour",
    "DepMinute",
    "DepTimeBlock",
    "Reporting_Airline",
    "Origin",
    "Dest",
    "Route",
    "Distance",
    "ScheduledDuration",
    "DistancePerMinute",
]

forbidden = {
    "DepTime", "DepDelay", "ArrDelay", "TaxiOut", "WheelsOff",
    "CarrierDelay", "WeatherDelay", "NASDelay",
    "SecurityDelay", "LateAircraftDelay"
}

assert not forbidden.intersection(features), "Potential leakage detected!"

print("\nModel features:")
print(features)

# 8. Data-quality checks
print(f"\nExact duplicate rows: {df.duplicated().sum():,}")
print(f"CRSDepTime == 2400: {(df['CRSDepTime'] == 2400).sum():,}")

print("\nMissing values in retained source columns:")
print(df[required_columns].isna().sum())

print("\nMonthly departure-delay rate (%):")
print(
    (df.groupby("FlightMonth", observed=True)["DepDel15"].mean() * 100)
    .round(2)
)

print(
    f"\nFinal DataFrame memory: "
    f"{df.memory_usage(deep=True).sum() / 1024**3:.2f} GB"
)

# 9. Save the processed dataset
df.to_parquet(output_path, index=False)

print(f"\nSaved: {output_path.resolve()}")
print(f"Parquet size: {output_path.stat().st_size / 1024**3:.2f} GB")
print(f"Total runtime: {time.perf_counter() - start_time:.1f}s")

month_1: 547,271 rows in 2.7s
month_10: 615,497 rows in 2.6s
month_11: 575,404 rows in 3.0s
month_12: 590,581 rows in 3.0s
month_2: 519,221 rows in 2.6s
month_3: 591,767 rows in 3.0s
month_4: 582,185 rows in 2.8s
month_5: 609,743 rows in 2.9s
month_6: 611,132 rows in 3.0s
month_7: 634,613 rows in 3.3s
month_8: 619,025 rows in 3.0s
month_9: 582,622 rows in 2.9s

Combined rows: 7,079,061
Selected columns: 9
Initial DataFrame memory: 1.67 GB

Rows with missing target by month:
Month
1     19858
2      2876
3      4945
4      3848
5      7976
6      7456
7     17857
8     12083
9      3421
10     6202
11     2532
12     3916
dtype: int64

Rows removed: 92,970
Rows after cleaning: 6,986,091

Model features:
['FlightMonth', 'FlightDay', 'FlightDayOfWeek', 'DepHour', 'DepMinute', 'DepTimeBlock', 'Reporting_Airline', 'Origin', 'Dest', 'Route', 'Distance', 'ScheduledDuration', 'DistancePerMinute']

Exact duplicate rows: 46
CRSDepTime == 2400: 1

Missing values in retained source columns:
Flight

### Step 2: Verify the Saved Dataset

Reload the Parquet file to confirm it can be used independently of the original CSV files. Check the saved shape, date range, target distribution, and monthly delay rates. This establishes our reproducible starting point for temporal validation.

In [2]:
import time
import pandas as pd
from pathlib import Path

parquet_path = Path("../data/processed/flights_2024.parquet")

start = time.perf_counter()
df = pd.read_parquet(parquet_path)
load_time = time.perf_counter() - start

print("Shape:", df.shape)
print("Date range:", df["FlightDate"].min(), "to", df["FlightDate"].max())
print(f"Load time: {load_time:.2f} seconds")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1024**3:.2f} GB")
print(f"File size: {parquet_path.stat().st_size / 1024**2:.1f} MB")

print("\nTarget distribution:")
print(df["DepDel15"].value_counts(normalize=True).sort_index().mul(100).round(2))

print("\nMonthly delay rate (%):")
print(
    df.groupby("FlightMonth", observed=True)["DepDel15"]
      .mean().mul(100).round(2)
)

Shape: (6986091, 18)
Date range: 2024-01-01 00:00:00 to 2024-12-31 00:00:00
Load time: 0.79 seconds
Memory: 0.28 GB
File size: 83.0 MB

Target distribution:
DepDel15
0    79.34
1    20.66
Name: proportion, dtype: float64

Monthly delay rate (%):
FlightMonth
1     23.18
2     15.50
3     20.84
4     18.90
5     25.66
6     25.33
7     29.52
8     23.07
9     15.08
10    13.61
11    14.41
12    21.65
Name: DepDel15, dtype: float64


### Step 3: Create a Temporal Train/Test Split

Randomly splitting flights can mix future and past records, making evaluation less representative of real deployment. We will train on January–September 2024 and reserve October–December 2024 as an unseen test period.

We will compare delay prevalence between the periods before training. Threshold selection and model tuning must use a validation slice from the training period, never the final test set.

In [3]:
# Sort chronologically
df = df.sort_values(["FlightDate", "CRSDepTime"]).reset_index(drop=True)

# Define date cutoffs
train_end = pd.Timestamp("2024-10-01")
test_end = pd.Timestamp("2025-01-01")

train_df = df[df["FlightDate"] < train_end].copy()
test_df = df[
    (df["FlightDate"] >= train_end)
    & (df["FlightDate"] < test_end)
].copy()

print("Training period:")
print(train_df["FlightDate"].min(), "to", train_df["FlightDate"].max())
print("Rows:", f"{len(train_df):,}")
print("Delay rate:", f"{train_df['DepDel15'].mean():.2%}")

print("\nTest period:")
print(test_df["FlightDate"].min(), "to", test_df["FlightDate"].max())
print("Rows:", f"{len(test_df):,}")
print("Delay rate:", f"{test_df['DepDel15'].mean():.2%}")

print("\nPrevalence difference (test - train):",
      f"{(test_df['DepDel15'].mean() - train_df['DepDel15'].mean()) * 100:.2f} percentage points")

Training period:
2024-01-01 00:00:00 to 2024-09-30 00:00:00
Rows: 5,217,259
Delay rate: 22.06%

Test period:
2024-10-01 00:00:00 to 2024-12-31 00:00:00
Rows: 1,768,832
Delay rate: 16.53%

Prevalence difference (test - train): -5.52 percentage points


### Step 4: Separate Training and Validation Data

Use January–August 2024 to fit models and September to select hyperparameters or a decision threshold. The October–December test set remains untouched until final evaluation.

This preserves chronological order and reduces the risk of choosing a model or threshold based on test-set performance.

In [4]:
validation_start = pd.Timestamp("2024-09-01")
train_start = pd.Timestamp("2024-01-01")

fit_df = df[
    (df["FlightDate"] >= train_start)
    & (df["FlightDate"] < validation_start)
].copy()

val_df = df[
    (df["FlightDate"] >= validation_start)
    & (df["FlightDate"] < train_end)
].copy()

print("Fit period:", fit_df["FlightDate"].min(), "to", fit_df["FlightDate"].max())
print("Fit rows:", f"{len(fit_df):,}")
print("Fit delay rate:", f"{fit_df['DepDel15'].mean():.2%}")

print("\nValidation period:", val_df["FlightDate"].min(), "to", val_df["FlightDate"].max())
print("Validation rows:", f"{len(val_df):,}")
print("Validation delay rate:", f"{val_df['DepDel15'].mean():.2%}")

print("\nFinal test rows:", f"{len(test_df):,}")

Fit period: 2024-01-01 00:00:00 to 2024-08-31 00:00:00
Fit rows: 4,638,058
Fit delay rate: 22.93%

Validation period: 2024-09-01 00:00:00 to 2024-09-30 00:00:00
Validation rows: 579,201
Validation delay rate: 15.08%

Final test rows: 1,768,832


### Step 5: Prevalence Baseline

A model should outperform a simple reference point before we consider it useful. The prevalence baseline reflects how common delayed flights are in each period. Because PR-AUC depends on positive-class prevalence, we will compare each model's PR-AUC against the prevalence of the same evaluation set.

This baseline does not learn to distinguish individual flights; it tells us the minimum reference performance to beat.

In [5]:
from sklearn.metrics import average_precision_score

# Constant scores: every flight gets the same prediction score
val_y = val_df["DepDel15"]

val_constant_scores = np.full(len(val_y), val_y.mean())

val_baseline_pr_auc = average_precision_score(
    val_y,
    val_constant_scores
)

test_y = test_df["DepDel15"]

test_constant_scores = np.full(len(test_y), test_y.mean())

test_baseline_pr_auc = average_precision_score(
    test_y,
    test_constant_scores
)

print(f"Validation prevalence: {val_y.mean():.4f}")
print(f"Validation baseline PR-AUC: {val_baseline_pr_auc:.4f}")

print(f"\nTest prevalence: {test_y.mean():.4f}")
print(f"Test baseline PR-AUC: {test_baseline_pr_auc:.4f}")

Validation prevalence: 0.1508
Validation baseline PR-AUC: 0.1508

Test prevalence: 0.1653
Test baseline PR-AUC: 0.1653


## Step 6 — Simple hour-of-day baseline

Before XGBoost, let’s see whether scheduled departure hour alone predicts delays better than the prevalence baseline.

This baseline calculates the historical delay rate for each scheduled departure hour using the fit data only. We then apply those rates to validation and test. No future labels are used to calculate the rates.

In [6]:
from sklearn.metrics import average_precision_score

hour_delay_rate = fit_df.groupby("DepHour")["DepDel15"].mean()

val_hour_scores = val_df["DepHour"].map(hour_delay_rate).fillna(
    fit_df["DepDel15"].mean()
)

test_hour_scores = test_df["DepHour"].map(hour_delay_rate).fillna(
    fit_df["DepDel15"].mean()
)

val_hour_pr_auc = average_precision_score(
    val_df["DepDel15"], val_hour_scores
)

test_hour_pr_auc = average_precision_score(
    test_df["DepDel15"], test_hour_scores
)

print(f"Validation prevalence baseline: {val_df['DepDel15'].mean():.4f}")
print(f"Validation hour baseline PR-AUC: {val_hour_pr_auc:.4f}")

print(f"\nTest prevalence baseline: {test_df['DepDel15'].mean():.4f}")
print(f"Test hour baseline PR-AUC: {test_hour_pr_auc:.4f}")

Validation prevalence baseline: 0.1508
Validation hour baseline PR-AUC: 0.2067

Test prevalence baseline: 0.1653
Test hour baseline PR-AUC: 0.2154



## Step 7 — Set up XGBoost

We’ll start with the existing 13 features and avoid one-hot encoding the high-cardinality `Route` column, which could create a huge matrix. First, check your installed XGBoost version so we choose a compatible categorical strategy.

We’ll check whether XGBoost is available and identify its version before choosing native categorical handling or a simpler encoding approach.

In [7]:
import xgboost as xgb

print("XGBoost version:", xgb.__version__)

XGBoost version: 3.4.1


## 7. Prepare Data for XGBoost

### Goal
Prepare the training, validation, and test datasets for a fair temporal evaluation.

### Feature strategy
- Keep the existing 13 features from the V1 benchmark.
- Use XGBoost's native categorical support for `Reporting_Airline`, `Origin`, `Dest`, `Route`, and `DepTimeBlock`.
- Avoid one-hot encoding high-cardinality features such as `Route`, which could create a very large feature matrix.

### Evaluation design
- **Fit:** January–August 2024
- **Validation:** September 2024
- **Final test:** October–December 2024

The model will be fitted only on the fit period. Validation will be used for model selection and threshold tuning. The final test set will remain untouched until the model is finalized.

### Leakage prevention
Only features known at least two hours before scheduled departure will be used. Actual departure times, actual delays, arrival information, and delay-cause fields must not enter the feature matrix.

In [8]:
import xgboost as xgb
from sklearn.metrics import average_precision_score, roc_auc_score

target = "DepDel15"

categorical_features = [
    "DepTimeBlock",
    "Reporting_Airline",
    "Origin",
    "Dest",
    "Route",
]

# Copy the feature list so we preserve the original definition.
model_features = features.copy()

X_fit = fit_df[model_features].copy()
y_fit = fit_df[target].astype("int8")

X_val = val_df[model_features].copy()
y_val = val_df[target].astype("int8")

X_test = test_df[model_features].copy()
y_test = test_df[target].astype("int8")

# Set consistent categorical dtypes across all three datasets.
for col in categorical_features:
    categories = fit_df[col].astype("category").cat.categories

    X_fit[col] = pd.Categorical(X_fit[col], categories=categories)
    X_val[col] = pd.Categorical(X_val[col], categories=categories)
    X_test[col] = pd.Categorical(X_test[col], categories=categories)

print("Fit:", X_fit.shape, "| delay rate:", round(y_fit.mean(), 4))
print("Validation:", X_val.shape, "| delay rate:", round(y_val.mean(), 4))
print("Test:", X_test.shape, "| delay rate:", round(y_test.mean(), 4))
print("Categorical features:", categorical_features)

Fit: (4638058, 13) | delay rate: 0.2293
Validation: (579201, 13) | delay rate: 0.1508
Test: (1768832, 13) | delay rate: 0.1653
Categorical features: ['DepTimeBlock', 'Reporting_Airline', 'Origin', 'Dest', 'Route']


## 8. Validate Model Features

### Goal
Identify missing values in the fit, validation, and test feature matrices before training.

### Why this matters
- `CRSDepTime == 2400` can produce `DepHour == 24`, which falls outside the current departure-time bins.
- Missing scheduled duration can produce a missing `DistancePerMinute`.
- XGBoost can handle missing numerical values, but we should understand where they occur.
- Categorical values absent from the fit-period categories may become missing in validation or test.

We will inspect the data first rather than silently deleting rows or filling values without justification.

In [9]:
for name, X in [
    ("Fit", X_fit),
    ("Validation", X_val),
    ("Test", X_test),
]:
    print(f"\n{name}:")
    missing = X.isna().sum()
    print(missing[missing > 0].sort_values(ascending=False))

print("\nRows with CRSDepTime == 2400:", (df["CRSDepTime"] == 2400).sum())
print("Rows with DepHour == 24:", (df["DepHour"] == 24).sum())


Fit:
DepTimeBlock         1
ScheduledDuration    1
DistancePerMinute    1
dtype: int64

Validation:
Series([], dtype: int64)

Test:
Series([], dtype: int64)

Rows with CRSDepTime == 2400: 1
Rows with DepHour == 24: 1


## 9. Handle Scheduled Midnight and Missing Duration

### Issue 1: Scheduled departure time `2400`
BTS contains one flight with `CRSDepTime = 2400`. The current feature engineering turns this into `DepHour = 24`, which does not fit our departure-time bins.

For modeling, we will normalize this one value to `0000` (midnight), setting `DepHour = 0`, `DepMinute = 0`, and `DepTimeBlock = Night`. This treats `2400` as the midnight boundary for time-of-day features without deleting the flight.

### Issue 2: Missing scheduled duration
One fit-period flight has a missing `ScheduledDuration`, which also makes `DistancePerMinute` missing. We will keep the row. XGBoost supports missing numerical values natively.

### Validation
After the correction, we will confirm that no model features are missing in the fit, validation, or test matrices except any numerical missing values we intentionally retain.

In [10]:
# Normalize the single 2400 scheduled-time row in each feature matrix.
for X in (X_fit, X_val, X_test):
    midnight = X["DepHour"].eq(24)

    X.loc[midnight, "DepHour"] = 0
    X.loc[midnight, "DepMinute"] = 0
    X.loc[midnight, "DepTimeBlock"] = "Night"

    # Restore the categorical dtype after assigning the category.
    X["DepTimeBlock"] = X["DepTimeBlock"].cat.add_categories(
        ["Night"]
    ) if "Night" not in X["DepTimeBlock"].cat.categories else X["DepTimeBlock"]

print("Fit missing values:")
print(X_fit.isna().sum()[X_fit.isna().sum() > 0])

print("\nValidation missing values:")
print(X_val.isna().sum()[X_val.isna().sum() > 0])

print("\nTest missing values:")
print(X_test.isna().sum()[X_test.isna().sum() > 0])

print("\nFit rows with DepHour == 24:", (X_fit["DepHour"] == 24).sum())

Fit missing values:
ScheduledDuration    1
DistancePerMinute    1
dtype: int64

Validation missing values:
Series([], dtype: int64)

Test missing values:
Series([], dtype: int64)

Fit rows with DepHour == 24: 0


## 10. Train the First XGBoost Model

### Objective
Train a baseline gradient-boosted tree model to rank flights by their likelihood of a 15+ minute departure delay.

### Configuration
- **Features:** The existing 13 V1 features.
- **Categorical data:** Native XGBoost categorical support.
- **Training period:** January–August 2024.
- **Validation period:** September 2024.
- **Early stopping:** Stop adding trees when validation performance stops improving.
- **Primary metric:** PR-AUC (`aucpr`), because delayed flights are the minority class.

### Evaluation safeguards
The model will not train on September validation labels or October–December test labels. We will use September to select the model, then evaluate the final model on the test period.

This is our first full-year model baseline, not yet a tuned or production-ready model.

In [ ]:
model = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    enable_categorical=True,
    n_estimators=1500,
    learning_rate=0.05,
    max_depth=8,
    min_child_weight=10,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=5,
    early_stopping_rounds=50,
    random_state=42,
    n_jobs=-1,
)

model.fit(
    X_fit,
    y_fit,
    eval_set=[(X_val, y_val)],
    verbose=100,
)

print("\nBest iteration:", model.best_iteration)
print("Best validation PR-AUC:", model.best_score)

[0]	validation_0-aucpr:0.23538


## 11. Evaluate the First XGBoost Model

### Goal
Compare XGBoost against our two baselines using PR-AUC and ROC-AUC.

### Metrics
- **PR-AUC:** Measures how well the model ranks delayed flights, focusing on the minority class.
- **ROC-AUC:** Measures how well the model separates delayed from non-delayed flights across thresholds.
- **Prevalence lift:** PR-AUC divided by the delayed-flight rate. Values above 1 indicate improvement over the prevalence baseline.

### Evaluation rules
- Use September validation results to assess the model and make modeling decisions.
- Evaluate October–December test results as a held-out check.
- Do not tune the model or decision threshold based on test results.

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score

val_scores = model.predict_proba(X_val)[:, 1]
test_scores = model.predict_proba(X_test)[:, 1]

results = pd.DataFrame([
    {
        "Dataset": "Validation",
        "PR-AUC": average_precision_score(y_val, val_scores),
        "ROC-AUC": roc_auc_score(y_val, val_scores),
        "Prevalence": y_val.mean(),
    },
    {
        "Dataset": "Test",
        "PR-AUC": average_precision_score(y_test, test_scores),
        "ROC-AUC": roc_auc_score(y_test, test_scores),
        "Prevalence": y_test.mean(),
    },
])

results["PR-AUC Lift"] = results["PR-AUC"] / results["Prevalence"]

print(results.round(4).to_string(index=False))

   Dataset  PR-AUC  ROC-AUC  Prevalence  PR-AUC Lift
Validation  0.2436   0.6616      0.1508       1.6156
      Test  0.2437   0.6379      0.1653       1.4740


## 13. Tune the Classification Threshold

### Goal
Choose a probability threshold using the September validation set, then apply that fixed threshold to the October–December test set.

### Method
We will select the threshold that maximizes the validation F1 score. F1 balances precision and recall.

- **Precision:** Of the flights flagged as delayed, how many were delayed?
- **Recall:** Of all delayed flights, how many did we flag?
- **Threshold:** The predicted probability above which a flight is classified as delayed.

This threshold is a starting operating point, not automatically the best business choice. In practice, the cost of missed delays versus unnecessary alerts determines the preferred balance.

The test set will not be used to select the threshold.

In [ ]:
import numpy as np
from sklearn.metrics import precision_recall_curve, precision_score, recall_score, f1_score

precision, recall, thresholds = precision_recall_curve(y_val, val_scores)

# The final precision/recall entries have no matching threshold.
f1_values = (
    2 * precision[:-1] * recall[:-1]
    / (precision[:-1] + recall[:-1] + 1e-12)
)

best_idx = np.argmax(f1_values)
best_threshold = thresholds[best_idx]

val_pred = (val_scores >= best_threshold).astype("int8")
test_pred = (test_scores >= best_threshold).astype("int8")

print(f"Validation-selected threshold: {best_threshold:.4f}")

for name, y_true, y_pred in [
    ("Validation", y_val, val_pred),
    ("Test", y_test, test_pred),
]:
    print(
        f"{name}: "
        f"Precision={precision_score(y_true, y_pred, zero_division=0):.4f}, "
        f"Recall={recall_score(y_true, y_pred, zero_division=0):.4f}, "
        f"F1={f1_score(y_true, y_pred, zero_division=0):.4f}"
    )

Validation-selected threshold: 0.2375
Validation: Precision=0.2269, Recall=0.5973, F1=0.3289
Test: Precision=0.2322, Recall=0.5582, F1=0.3280


## 1. Investigate Early Stopping

### Goal
Understand why XGBoost selected iteration 4 as its best model.

### In simple words
XGBoost builds trees one after another. After each tree, it checks how well the model performs on September's validation data.

- If validation performance improves, it keeps the better model.
- If performance stops improving for 50 rounds, training stops.
- The best iteration is where validation PR-AUC was highest, not necessarily where training stopped.

We will inspect the saved validation history to confirm this behavior before making any changes.

In [ ]:
val_history = model.evals_result()["validation_0"]["aucpr"]

best_idx = int(np.argmax(val_history))

print("Rounds recorded:", len(val_history))
print("Best iteration (0-based):", best_idx)
print(f"Best validation PR-AUC: {val_history[best_idx]:.5f}")
print(f"Score after first tree: {val_history[0]:.5f}")
print(f"Score at final recorded round: {val_history[-1]:.5f}")

print("\nFirst 10 rounds:")
for i, score in enumerate(val_history[:10]):
    print(f"Round {i}: {score:.5f}")

Rounds recorded: 55
Best iteration (0-based): 4
Best validation PR-AUC: 0.24367
Score after first tree: 0.23538
Score at final recorded round: 0.23936

First 10 rounds:
Round 0: 0.23538
Round 1: 0.23897
Round 2: 0.24082
Round 3: 0.24148
Round 4: 0.24367
Round 5: 0.24323
Round 6: 0.24243
Round 7: 0.24280
Round 8: 0.24242
Round 9: 0.24136


## 2. Inspect Feature Importance

### Goal
Find which features XGBoost relied on most when predicting departure delays.

### In simple words
Think of feature importance as asking the model: “Which information helped you make your predictions?”

We'll inspect **gain importance**, which measures how much each feature improved the model's tree splits on average, weighted by how often those splits were used.

Important: this shows how the model used a feature, not whether that feature causes delays. Correlated features can also share importance.

In [ ]:
importance = (
    pd.DataFrame({
        "Feature": model.feature_names_in_,
        "Gain": [
            model.get_booster().get_score(
                importance_type="gain"
            ).get(feature, 0)
            for feature in model.feature_names_in_
        ],
    })
    .sort_values("Gain", ascending=False)
    .reset_index(drop=True)
)

print(importance.round(4).to_string(index=False))

          Feature      Gain
     DepTimeBlock 7878.0747
          DepHour 2405.6838
Reporting_Airline  902.5478
      FlightMonth  331.0247
           Origin  326.1075
            Route  213.3592
        FlightDay  205.0675
             Dest  147.3586
  FlightDayOfWeek  112.1226
         Distance   47.2514
DistancePerMinute   16.3722
        DepMinute   10.3838
ScheduledDuration    9.0842


## 3. Final Model Sanity Checks

### Goal
Confirm that the model uses only approved features and that our evaluation still follows the time-based split.

### Checks
- Confirm that no actual departure or arrival outcomes appear in the model features.
- Confirm the fit, validation, and test dates do not overlap.
- Confirm that our selected threshold came from September validation data.
- Record the final test metrics without using the test set to tune the model.

These checks help catch data leakage and evaluation mistakes before we report the results.

In [ ]:
forbidden_terms = [
    "DepTime", "DepDelay", "ArrTime", "ArrDelay",
    "TaxiOut", "WheelsOff", "WheelsOn",
    "CarrierDelay", "WeatherDelay", "NASDelay",
    "SecurityDelay", "LateAircraftDelay",
]

leakage_features = [
    f for f in model_features
    if any(term in f for term in forbidden_terms)
]

print("Potential leakage features:", leakage_features)
print("Fit ends:", fit_df["FlightDate"].max().date())
print("Validation:", val_df["FlightDate"].min().date(), "to",
      val_df["FlightDate"].max().date())
print("Test:", test_df["FlightDate"].min().date(), "to",
      test_df["FlightDate"].max().date())
print(f"Validation-selected threshold: {best_threshold:.4f}")
print(f"Final test PR-AUC: {average_precision_score(y_test, test_scores):.4f}")
print(f"Final test ROC-AUC: {roc_auc_score(y_test, test_scores):.4f}")
print(f"Final test precision: {precision_score(y_test, test_pred):.4f}")
print(f"Final test recall: {recall_score(y_test, test_pred):.4f}")
print(f"Final test F1: {f1_score(y_test, test_pred):.4f}")

Potential leakage features: ['DepTimeBlock']
Fit ends: 2024-08-31
Validation: 2024-09-01 to 2024-09-30
Test: 2024-10-01 to 2024-12-31
Validation-selected threshold: 0.2375
Final test PR-AUC: 0.2437
Final test ROC-AUC: 0.6379
Final test precision: 0.2322
Final test recall: 0.5582
Final test F1: 0.3280


## 1. Verify Feature Leakage

### Goal
Confirm that the model only uses information available at least two hours before scheduled departure.

### Why this matters
A model must not use information that becomes available only after a flight departs. Otherwise, it can appear accurate during testing but fail in real use.

We will compare feature names against an explicit list of forbidden columns. This avoids false alarms caused by matching partial names.

`DepTimeBlock` is allowed because it is derived from scheduled departure time (`CRSDepTime`). Actual departure time (`DepTime`) is forbidden.

In [ ]:
forbidden_features = {
    "DepTime",
    "DepDelay",
    "ArrTime",
    "ArrDelay",
    "TaxiOut",
    "WheelsOff",
    "WheelsOn",
    "CarrierDelay",
    "WeatherDelay",
    "NASDelay",
    "SecurityDelay",
    "LateAircraftDelay",
}

leakage_features = sorted(set(model_features) & forbidden_features)

print("Forbidden features found:", leakage_features)
print("Number of model features:", len(model_features))
print("DepTimeBlock included:", "DepTimeBlock" in model_features)

Forbidden features found: []
Number of model features: 13
DepTimeBlock included: True


## 2. Compare Time-Only and Full Models

### Goal
Find out whether the full feature set improves predictions beyond scheduled time-of-day information.

### Models
- **Time-only:** `DepTimeBlock`, `DepHour`, and `DepMinute`.
- **Full model:** All 13 features.

### Fair comparison
Both models use the same January–August fit data and September validation data. We compare validation PR-AUC and do not use the final test set to make this modeling decision.

If the full model performs better, the additional features provide useful predictive information beyond time of day alone.

In [ ]:
time_features = ["DepTimeBlock", "DepHour", "DepMinute"]

time_model = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    enable_categorical=True,
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=10,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=5,
    early_stopping_rounds=30,
    random_state=42,
    n_jobs=-1,
)

time_model.fit(
    X_fit[time_features],
    y_fit,
    eval_set=[(X_val[time_features], y_val)],
    verbose=False,
)

time_val_scores = time_model.predict_proba(X_val[time_features])[:, 1]

time_pr_auc = average_precision_score(y_val, time_val_scores)
full_pr_auc = average_precision_score(y_val, val_scores)

print(f"Time-only validation PR-AUC: {time_pr_auc:.4f}")
print(f"Full-model validation PR-AUC: {full_pr_auc:.4f}")
print(f"Time-only best iteration: {time_model.best_iteration}")
print(f"Full-model best iteration: {model.best_iteration}")
print(f"PR-AUC difference (full - time-only): {full_pr_auc - time_pr_auc:+.4f}")

Time-only validation PR-AUC: 0.2096
Full-model validation PR-AUC: 0.2436
Time-only best iteration: 54
Full-model best iteration: 4
PR-AUC difference (full - time-only): +0.0340


## 3. Diagnose Early Stopping

### Goal
Check whether the full model fits the training data much better than it performs on September validation data.

### In simple words
- **Training PR-AUC rises while validation PR-AUC falls:** a sign of overfitting.
- **Both scores stay low or flatten:** the model may have reached its useful limit with the current settings.
- **Validation improves again with more trees:** the earlier stopping behavior may need further investigation.

We will run a short diagnostic fit and record PR-AUC on both datasets. This is a diagnostic experiment, not a replacement for the current model or a final test evaluation.

In [ ]:
diagnostic_model = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    enable_categorical=True,
    n_estimators=30,
    learning_rate=0.05,
    max_depth=8,
    min_child_weight=10,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=5,
    random_state=42,
    n_jobs=-1,
)

diagnostic_model.fit(
    X_fit,
    y_fit,
    eval_set=[
        (X_fit, y_fit),
        (X_val, y_val),
    ],
    verbose=False,
)

history = diagnostic_model.evals_result()

for dataset_name in ("validation_0", "validation_1"):
    scores = history[dataset_name]["aucpr"]
    best_round = int(np.argmax(scores))
    print(
        f"{dataset_name}: "
        f"first={scores[0]:.4f}, "
        f"best={scores[best_round]:.4f} at round {best_round}, "
        f"last={scores[-1]:.4f}"
    )

validation_0: first=0.4034, best=0.4387 at round 29, last=0.4387
validation_1: first=0.2354, best=0.2437 at round 4, last=0.2395
